# 1.4 AirSim Drone Visual Perception

AirSim is an open-source simulator built on Unreal Engine. Its camera data acquisition capability is essential for drone development.


### Multiple Camera Types Supported

AirSim supports various common camera types, including scene cameras (for capturing color images from the drone's perspective), depth cameras (generating depth information to help drones perceive distance), and fisheye cameras (providing a wider field of view for panoramic data collection). Developers can easily select and configure the desired camera type in code based on their specific needs.

### Flexible Data Acquisition Configuration

Developers can fine-tune camera parameters such as image resolution (from low-resolution 320x240 to high-resolution 4K or higher), frame rate (adjustable based on task requirements, e.g., higher frame rates for real-time tracking tasks), and field of view angle. Additionally, data acquisition frequency can be specified - either at fixed time intervals or triggered by specific events - providing great flexibility for different application scenarios.

### Convenient Data Retrieval

At the code level, developers can retrieve camera data through simple function calls using the AirSim API. For example, in Python, use the airsim.MultirotorClient class methods to first connect to the AirSim simulator, then retrieve camera image data through corresponding commands. The data is returned in common image formats (PNG, JPEG for color images, or special formats for depth image data), making it convenient for subsequent analysis and processing with image processing libraries like OpenCV.



Both drones and cars in the AirSim platform have 5 cameras by default, differing only in position. This book focuses on drone cameras. The table below lists camera positions and names. Older versions of AirSim used numbers for camera naming; current AirSim is backward-compatible, so numbers still work.

| Camera Position  | ID                | Legacy ID     |
|------------------|-------------------|---------------|
| Front Center     | `"front_center"`  | `"0"`         |
| Front Right      | `"front_right"`   | `"1"`         |
| Front Left       | `"front_left"`    | `"2"`         |
| Bottom Center    | `"bottom_center"` | `"3"`         |
| Back Center      | `"back_center"`   | `"4"`         |

<img src="img/s1-4-1.png" width="600px" />


<img src="img/s1-4-2.png" width="600px" />

AirSim provides 8 different image types by default. The table below summarizes the name, description, and API call for each type.

| Image Type          | Description                                                              | API Call                          |
|---------------------|--------------------------------------------------------------------------|-----------------------------------|
| Scene               | Color image, visible light image of the scene                            | `airsim.ImageType.Scene`         |
| DepthPlanar         | Depth map, pixel values represent Euclidean distance to camera plane (meters) | `airsim.ImageType.DepthPlanar`    |
| DepthPerspective    | Depth map, pixel values represent distance under perspective projection  | `airsim.ImageType.DepthPerspective` |
| DepthVis            | Depth visualization, gradient from 0m (black) to 100m (white)           | `airsim.ImageType.DepthVis`       |
| DisparityNormalized | Normalized disparity map, float values (0-1), inversely proportional to depth | `airsim.ImageType.DisparityNormalized` |
| Segmentation        | Segmentation map, objects marked with unique colors (based on object IDs) | `airsim.ImageType.Segmentation`   |
| SurfaceNormals      | Surface normals map, RGB channels represent XYZ components of normal vectors | `airsim.ImageType.SurfaceNormals` |
| Infrared            | Infrared image, grayscale based on object temperature or preset IDs (simulates thermal imaging) | `airsim.ImageType.Infrared`       |

In [ ]:
!pip list

In [ ]:
!pip install opencv-python matplotlib
!pip install "matplotlib<3.9" -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install --upgrade pillow

In [ ]:
import sys
sys.path.append('..')
import airsim
import math
import numpy as np

In [ ]:
import time
import cv2
import matplotlib.pyplot as plt

# connect to the AirSim simulator
client = airsim.MultirotorClient()
client.confirmConnection()

# Enable API control (disabled by default)
client.enableApiControl(True)

# Arm/unlock, same process as real aircraft
client.armDisarm(True)

client.takeoffAsync().join()
print("Takeoff-------------------------")

In [ ]:
camera_name = '0'  # Camera position: front center
image_type = airsim.ImageType.Scene  # Image type: color image (Scene), Infrared

for i in range(3):
    response = client.simGetImage(camera_name, image_type, vehicle_name='')  # simGetImage API call
    if response:  # Binary image data
        # Convert to OpenCV image format
        img_bgr = cv2.imdecode(np.array(bytearray(response), dtype='uint8'), cv2.IMREAD_UNCHANGED)  # Read from binary data
        
        # Convert BGR to RGB since matplotlib displays RGB by default
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        # Display image in Jupyter Notebook
        plt.figure(figsize=(10, 6))
        plt.imshow(img_rgb)
        plt.axis('off')
        plt.show()
        
    time.sleep(1)

In [ ]:
# Land
client.landAsync().join()
print("Landing-------------------------")


# Lock
client.armDisarm(False)

# Release control
client.enableApiControl(False)

## Multithreading
AirSim uses a multithreaded mode. Note that in multithreading, each thread needs to create its own new client instance - threads cannot share a single client. Also note that within Python threads, you need to use a_cv2_imshow_thread to display images, as OpenCV may not be able to show images in some environments and can only save to files.

In [ ]:
!pip install a_cv2_imshow_thread 

In [ ]:
import threading
from a_cv2_imshow_thread import add_imshow_thread_to_cv2
add_imshow_thread_to_cv2()  # monkey patching


def show():
    client = airsim.MultirotorClient()
    camera_name = '0'  # Front center camera
    image_type = airsim.ImageType.Scene  # Color image: airsim.ImageType.Scene, Infrared
    for i in range(3):
        response = client.simGetImage(camera_name, image_type, vehicle_name='')  # simGetImage API call
        if response:
            print("len(response): ", len(response))
            # Convert to OpenCV image format
            img_bgr = cv2.imdecode(np.array(bytearray(response), dtype='uint8'), cv2.IMREAD_UNCHANGED)  # Read from binary data
            # Save as image file
            cv2.imwrite('scene{}.jpg'.format(i), img_bgr)  # Save as .jpg image file

            # Show output
            cv2.imshow("preview", img_bgr)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        print("get image")
        time.sleep(1)



# Connect to the AirSim simulator
client = airsim.MultirotorClient()


# Enable API control (disabled by default)
client.enableApiControl(True)

# Arm/unlock, same process as real aircraft
client.armDisarm(True)

client.takeoffAsync().join()


print("Takeoff-------------------------")

threading1 = threading.Thread(target=show)
threading1.setDaemon(True)
threading1.start()


# Land
client.landAsync().join()
print("Landing-------------------------")

# Lock
client.armDisarm(False)

# Release control
client.enableApiControl(False)


## Multiprocessing
It is recommended to use a dedicated process specifically for visual processing.

In [ ]:
# Connect to the AirSim simulator
client1 = airsim.MultirotorClient()
client1.confirmConnection()

# Enable API control (disabled by default)
client1.enableApiControl(True)

# Arm/unlock, same process as real aircraft
client1.armDisarm(True)

client1.takeoffAsync().join()
print("Takeoff-------------------------")

In [ ]:
# Connect to the AirSim simulator
client2 = airsim.MultirotorClient()

# No need to arm/unlock - just use it directly


camera_name = '0'  # Front center camera
image_type = airsim.ImageType.Scene  # Color image: airsim.ImageType.Scene, Infrared

for i in range(3):
    response = client2.simGetImage(camera_name, image_type, vehicle_name='')  # simGetImage API call
    if response:  # Binary image data
        # Convert to OpenCV image format
        img_bgr = cv2.imdecode(np.array(bytearray(response), dtype='uint8'), cv2.IMREAD_UNCHANGED)  # Read from binary data
        
        # Convert BGR to RGB since matplotlib displays RGB by default
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        # Display image in Jupyter Notebook
        plt.figure(figsize=(10, 6))
        plt.imshow(img_rgb)
        plt.axis('off')
        plt.show()
        
    time.sleep(1)

## AirSim Camera Detailed Configuration


### Locating the Configuration File

The AirSim configuration file is named `settings.json` and is typically located in the AirSim project root directory. If AirSim is integrated into an Unreal Engine project, the file path may differ - usually in the project's `Content/Settings` folder or under C:\Users\{username}\Documents\AirSim. Once you find this file, you can open it with any text editor (e.g., Notepad++, Sublime Text).

### Configuration File Structure and Camera Settings Location

`settings.json` is a JSON-formatted text file with multiple sections. Camera configuration is located under the `Vehicles` section. For example, if you have a drone named `Drone1`, its camera configuration will be under the `Drone1` object within `Vehicles`. The file structure looks roughly like this:


```
{
    "SettingsVersion": 1.2,
    "SimMode": "Multirotor",
    "Vehicles": {
        "Drone1": {
            "VehicleType": "SimpleFlight",
            "X": 0, "Y": 0, "Z": -5,
            "CameraDefaults": {
                "CaptureSettings": []
            },
            "Cameras": {
                "front_center": {}
            }
        }
    }
}
```

In the structure above, `CameraDefaults` sets default parameters for all cameras, while `Cameras` is used for individual camera configuration.

### Resolution Configuration

To set camera resolution in the configuration file, add the corresponding entry in the `CaptureSettings` array. For example, to set the `front_center` camera resolution to 1920x1080:



```
{
    "SettingsVersion": 1.2,
    "SimMode": "Multirotor",
    "Vehicles": {
        "Drone1": {
            "VehicleType": "SimpleFlight",
            "X": 0, "Y": 0, "Z": -5,
            "CameraDefaults": {
                "CaptureSettings": [
                    {
                        "Width": 1920,
                        "Height": 1080,
                        "ImageType": 0,
                        "PixelFormat": "Uncompressed"
                    }
                ]
            },
            "Cameras": {
                "front_center": {}
            }
        }
    }
}

```

Here, `Width` and `Height` specify image dimensions, `ImageType` 0 represents the scene camera (corresponding to `airsim.ImageType.Scene`), and `PixelFormat` specifies the pixel format - `Uncompressed` ensures image quality but produces larger data, while `Compressed` reduces data size with potential quality loss.

### Frame Rate Configuration

Frame rate is also configured in `CaptureSettings`. To set the frame rate to 25 FPS:



```
{
    "SettingsVersion": 1.2,
    "SimMode": "Multirotor",
    "Vehicles": {
        "Drone1": {
            "VehicleType": "SimpleFlight",
            "X": 0, "Y": 0, "Z": -5,
            "CameraDefaults": {
                "CaptureSettings": [
                    {
                        "Width": 1920,
                        "Height": 1080,
                        "ImageType": 0,
                        "PixelFormat": "Uncompressed",
                        "Framerate": 25
                    }
                ]
            },
            "Cameras": {
                "front_center": {}
            }
        }
    }
}

```

Simply add the `Framerate` field and set it to the desired value.

### Field of View Configuration

Field of view (FOV) is also set in `CaptureSettings`. To set the horizontal FOV to 70 degrees:



```
{
    "SettingsVersion": 1.2,
    "SimMode": "Multirotor",
    "Vehicles": {
        "Drone1": {
            "VehicleType": "SimpleFlight",
            "X": 0, "Y": 0, "Z": -5,
            "CameraDefaults": {
                "CaptureSettings": [
                    {
                        "Width": 1920,
                        "Height": 1080,
                        "ImageType": 0,
                        "PixelFormat": "Uncompressed",
                        "Framerate": 25,
                        "FOV_Degrees": 70
                    }
                ]
            },
            "Cameras": {
                "front_center": {}
            }
        }
    }
}
```

Add the `FOV_Degrees` field and set the angle value to adjust the field of view.

### Camera Type Configuration

The `ImageType` field in the configuration file specifies the camera type. For example, to configure a depth camera (corresponding to `airsim.ImageType.DepthPlanner`), change `ImageType` to 2 (the depth camera type value in AirSim):



```
{
    "SettingsVersion": 1.2,
    "SimMode": "Multirotor",
    "Vehicles": {
        "Drone1": {
            "VehicleType": "SimpleFlight",
            "X": 0, "Y": 0, "Z": -5,
            "CameraDefaults": {
                "CaptureSettings": [
                    {
                        "Width": 1920,
                        "Height": 1080,
                        "ImageType": 2,
                        "PixelFormat": "Uncompressed",
                        "Framerate": 25,
                        "FOV_Degrees": 70
                    }
                ]
            },
            "Cameras": {
                "front_center": {}
            }
        }
    }
}

```

To configure a fisheye camera, besides setting `ImageType` (assuming fisheye also uses `airsim.ImageType.Scene` with type value 0), you may need additional fisheye-specific parameters such as distortion coefficients. This configuration is relatively complex and may vary by specific requirements and AirSim version. Generally, add related parameters under the camera configuration in the `Cameras` object:



```
{
    "SettingsVersion": 1.2,
    "SimMode": "Multirotor",
    "Vehicles": {
        "Drone1": {
            "VehicleType": "SimpleFlight",
            "X": 0, "Y": 0, "Z": -5,
            "CameraDefaults": {
                "CaptureSettings": [
                    {
                        "Width": 1920,
                        "Height": 1080,
                        "ImageType": 0,
                        "PixelFormat": "Uncompressed",
                        "Framerate": 25,
                        "FOV_Degrees": 70
                    }
                ]
            },
            "Cameras": {
                "front_center": {
                    "Fisheye": true,
                    "FisheyeDistortion": [0.1, 0.2, 0.3, 0.4] // Example distortion coefficients, adjust as needed
                }
            }
        }
    }
}
```

In the example above, setting `Fisheye` to `true` indicates a fisheye camera, and the `FisheyeDistortion` array sets distortion coefficients to adjust the fisheye image distortion effect.

After modifying `settings.json`, save the file and restart the AirSim simulator for the camera parameters to take effect. In code, you no longer need to set these parameters through function calls - AirSim will automatically read the settings from the configuration file.




Configuration reference documentation:
1. https://microsoft.github.io/AirSim/sensors/
2. https://zhaoxuhui.top/blog/2021/12/01/airsim-note3-settings.html